In [1]:
import pandas as pd
df = pd.read_csv('./data/nomenclature/7_cellosaurus.csv')  # or read_table for TSV
print(df.shape)
print(df.dtypes)
print(df.head(20).to_string())
print(df.isnull().sum())

(152231, 17)
Identifier (cell line name)       object
Accession (CVCL_xxxx)             object
Secondary accession number(s)     object
Synonyms                          object
Cross-references                  object
References identifiers            object
Web pages                         object
Comments                          object
STR profile data                  object
Diseases                          object
Species of origin                 object
Hierarchy                         object
Originate from same individual    object
Sex of cell                       object
Age of donor at sampling          object
Category                          object
Date (entry history)              object
dtype: object
   Identifier (cell line name) Accession (CVCL_xxxx) Secondary accession number(s)                                                                   Synonyms                                                                                Cross-references                       

In [2]:
print("Total rows:", len(df))
print("Unique CVCL accessions:", df['Accession (CVCL_xxxx)'].nunique())
print("Any duplicate accessions:", df['Accession (CVCL_xxxx)'].duplicated().sum())

Total rows: 152231
Unique CVCL accessions: 152231
Any duplicate accessions: 0


In [ ]:
# human = df[df['Species of origin'].str.contains('Homo sapiens', na=False)]
human = df[df['Species of origin'].str.contains('9606', na=False)]
print("Human cell lines:", len(human))
print("Non-human:", len(df) - len(human))

Human cell lines: 112768
Non-human: 39463


In [4]:
has_synonyms = df['Synonyms'].notna()
print("Lines with synonyms:", has_synonyms.sum())
print("Lines without synonyms:", (~has_synonyms).sum())

# Average number of synonyms per line
synonym_counts = df['Synonyms'].dropna().str.split(';').str.len()
print("Average synonyms per line:", synonym_counts.mean().round(2))
print("Max synonyms for one line:", synonym_counts.max())
print("Most aliased line:")
print(df.loc[synonym_counts.idxmax(), ['Identifier (cell line name)', 'Synonyms']])

Lines with synonyms: 72386
Lines without synonyms: 79845
Average synonyms per line: 1.55
Max synonyms for one line: 34
Most aliased line:
Identifier (cell line name)                                         P3X63Ag8.653
Synonyms                       P3-X63-Ag8.653; P3-X63-Ag8-653; P3-x63-Ag8 653...
Name: 124950, dtype: object


In [5]:
hela = df[df['Identifier (cell line name)'].str.contains('HeLa', case=False, na=False) |
          df['Synonyms'].str.contains('HeLa', case=False, na=False)]
print("HeLa-related entries:", len(hela))
print(hela[['Identifier (cell line name)', 'Accession (CVCL_xxxx)', 'Synonyms']].to_string())

HeLa-related entries: 1393
                                  Identifier (cell line name) Accession (CVCL_xxxx)                                                                                                                                                               Synonyms
1330                       1A3 [Mouse hybridoma against HeLa]             CVCL_J099                                                                                                                                                                    NaN
8670                                       Abcam HeLa AAK1 KO             CVCL_B1IJ                                                                                                                                               Human AAK1 knockout HeLa
8671                                      Abcam HeLa ABCC1 KO             CVCL_B1IK                                                                                                                                       Hu

In [6]:
known_lines = ['A549', 'MCF7', 'HeLa', 'HEK293', 'SKBR3', 'SW480', 'H1299', 'A431']
for line in known_lines:
    match = df[df['Identifier (cell line name)'].str.contains(line, case=False, na=False) |
               df['Synonyms'].str.contains(line, case=False, na=False)]
    print(f"{line}: {len(match)} entries found")
    if len(match) > 0:
        print(match[['Identifier (cell line name)', 'Accession (CVCL_xxxx)']].head(3).to_string())
    print()

A549: 1198 entries found
     Identifier (cell line name) Accession (CVCL_xxxx)
5304                       A-549             CVCL_0023
5621                A549 ATGL-KO             CVCL_UJ32
5622                 A549 B2M KO             CVCL_C3CC

MCF7: 617 entries found
      Identifier (cell line name) Accession (CVCL_xxxx)
10097        Abcam MCF-7 ABCC1 KO             CVCL_B8S1
10098         Abcam MCF-7 ABL2 KO             CVCL_B8S2
10099        Abcam MCF-7 ACKR3 KO             CVCL_B8S3

HeLa: 1393 entries found
             Identifier (cell line name) Accession (CVCL_xxxx)
1330  1A3 [Mouse hybridoma against HeLa]             CVCL_J099
8670                  Abcam HeLa AAK1 KO             CVCL_B1IJ
8671                 Abcam HeLa ABCC1 KO             CVCL_B1IK

HEK293: 1437 entries found
     Identifier (cell line name) Accession (CVCL_xxxx)
2143                  293-IL-1RI             CVCL_9831
2151                     293-VnR             CVCL_M907
2165                 293/CRE-Luc   

In [7]:
print(df['Category'].value_counts().head(10))

Category
Transformed cell line                   50625
Cancer cell line                        36263
Embryonic stem cell                     16478
Induced pluripotent stem cell           16381
Finite cell line                        12263
Hybridoma                               11013
Spontaneously immortalized cell line     6449
Telomerase immortalized cell line         833
Hybrid cell line                          747
Conditionally immortalized cell line      375
Name: count, dtype: int64


In [8]:
key_cols = ['Identifier (cell line name)', 'Accession (CVCL_xxxx)', 
            'Synonyms', 'Species of origin', 'Category', 'Diseases']
print(df[key_cols].isnull().sum())
print("\nMissing % on key columns:")
print((df[key_cols].isnull().sum() / len(df) * 100).round(1))

Identifier (cell line name)        1
Accession (CVCL_xxxx)              0
Synonyms                       79845
Species of origin                  0
Category                           0
Diseases                       81241
dtype: int64

Missing % on key columns:
Identifier (cell line name)     0.0
Accession (CVCL_xxxx)           0.0
Synonyms                       52.4
Species of origin               0.0
Category                        0.0
Diseases                       53.4
dtype: float64


In [10]:
import pandas as pd

df = pd.read_csv('./data/gene expression/2_DepMap_OmicsExpressionAllGenesTPMLogp1Profile.csv', index_col=0)
print("Shape:", df.shape)
print("\nFirst 5 row indices (cell line IDs):")
print(df.index[:5].tolist())
print("\nFirst 5 column headers:")
print(df.columns[:5].tolist())
print("\nLast 5 column headers:")
print(df.columns[-5:].tolist())

Shape: (1495, 53961)

First 5 row indices (cell line IDs):
['PR-AdBjpG', 'PR-I2AzwG', 'PR-5ekAAC', 'PR-I21681', 'PR-i9DRP1']

First 5 column headers:
['TSPAN6 (ENSG00000000003)', 'TNMD (ENSG00000000005)', 'DPM1 (ENSG00000000419)', 'SCYL3 (ENSG00000000457)', 'C1orf112 (ENSG00000000460)']

Last 5 column headers:
['ENSG00000288721', 'ENSG00000288722', 'ENSG00000288723', 'ENSG00000288724', 'ENSG00000288725']


In [11]:
df.head()

,TSPAN6 (ENSG00000000003),TNMD (ENSG00000000005),DPM1 (ENSG00000000419),SCYL3 (ENSG00000000457),C1orf112 (ENSG00000000460),FGR (ENSG00000000938),CFH (ENSG00000000971),FUCA2 (ENSG00000001036),GCLC (ENSG00000001084),NFYA (ENSG00000001167),...,ENSG00000288714,ENSG00000288717,ENSG00000288718,ENSG00000288719,ENSG00000288720,ENSG00000288721,ENSG00000288722,ENSG00000288723,ENSG00000288724,ENSG00000288725
PR-AdBjpG,4.331992,0.000000,7.364660,2.792855,4.471187,0.028569,1.226509,3.044394,6.500005,4.739848,...,0.000000,0.536053,0.000000,0.028569,0.176323,0.992768,2.797013,0.000000,0.0,0.000000
PR-I2AzwG,4.567424,0.584963,7.106641,2.543496,3.504620,0.000000,0.189034,3.813525,4.221877,3.481557,...,0.000000,0.879706,0.000000,0.014355,0.014355,0.432959,2.972693,0.056584,0.0,0.070389
PR-5ekAAC,3.150560,0.000000,7.379118,2.333424,4.228049,0.056584,1.310340,6.687201,3.682573,3.273516,...,0.028569,0.000000,0.084064,0.000000,0.097611,0.367371,1.695994,0.084064,0.0,0.000000
PR-I21681,5.085340,0.000000,7.154211,2.545968,3.084064,0.000000,5.868390,6.165309,4.489928,3.956986,...,0.000000,0.000000,0.070389,0.000000,0.176323,0.411426,3.921246,0.028569,0.0,0.000000
PR-i9DRP1,6.729417,0.000000,6.537917,2.456806,3.867896,0.799087,7.208478,5.570159,7.127117,4.568032,...,0.000000,0.000000,0.201634,0.028569,0.137504,0.678072,4.418190,0.000000,0.0,0.000000


In [12]:
import numpy as np

print("Min value across entire matrix:", df.values.min())
print("Max value across entire matrix:", df.values.max())
print("Mean value:", df.values.mean().round(4))
print("Median value:", np.median(df.values).round(4))

# Sample 5 values from EGFR column if it exists
egfr_cols = [c for c in df.columns if 'EGFR' in c]
print("\nEGFR column name(s):", egfr_cols[:3])
if egfr_cols:
    print("\nEGFR expression sample (10 cell lines):")
    print(df[egfr_cols[0]].head(10))

Min value across entire matrix: 0.0
Max value across entire matrix: 17.783669710269233
Mean value: 1.0888
Median value: 0.0144

EGFR column name(s): ['EGFR (ENSG00000146648)', 'EGFR-AS1 (ENSG00000224057)']

EGFR expression sample (10 cell lines):
PR-AdBjpG    6.179511
PR-I2AzwG    1.815575
PR-5ekAAC    1.372952
PR-I21681    4.767125
PR-i9DRP1    5.064366
PR-llpknG    3.617063
PR-feSGd6    2.754888
PR-z36vet    0.201634
PR-7WaDe1    5.558880
PR-DPUwIm    4.566206
Name: EGFR (ENSG00000146648), dtype: float64


In [13]:
total_cells = df.shape[0] * df.shape[1]
missing = df.isnull().sum().sum()
zeros = (df == 0).sum().sum()

print("Total cells:", total_cells)
print("Missing (NaN/blank):", missing)
print("Missing %:", round(missing / total_cells * 100, 4))
print("Zero values:", zeros)
print("Zero %:", round(zeros / total_cells * 100, 4))
print("\nAre missing and zero the same thing here?", missing == 0)

Total cells: 80671695
Missing (NaN/blank): 0
Missing %: 0.0
Zero values: 40209561
Zero %: 49.8435

Are missing and zero the same thing here? True


In [15]:
print("Cell lines in expression file:", df.shape[0])
print("Sample of ACH IDs:")
print(df.index[:10].tolist())

# Check ACH ID format
all_ach = all(str(idx).startswith('ACH-') for idx in df.index)
print("\nAll row indices start with ACH-:", all_ach)

# Load sample_info and compare
sample_info = pd.read_csv('./data/nomenclature/9_DepMap_sample_info.csv')
print("\nCell lines in sample_info:", len(sample_info))
print("sample_info ID column name:", sample_info.columns[0])

Cell lines in expression file: 1495
Sample of ACH IDs:
['PR-AdBjpG', 'PR-I2AzwG', 'PR-5ekAAC', 'PR-I21681', 'PR-i9DRP1', 'PR-llpknG', 'PR-feSGd6', 'PR-z36vet', 'PR-7WaDe1', 'PR-DPUwIm']

All row indices start with ACH-: False

Cell lines in sample_info: 1840
sample_info ID column name: DepMap_ID


In [16]:
expr_ids = set(df.index)
info_ids = set(sample_info.iloc[:, 0])

print("In expression but NOT in sample_info:", len(expr_ids - info_ids))
print("In sample_info but NOT in expression:", len(info_ids - expr_ids))
print("In both:", len(expr_ids & info_ids))

In expression but NOT in sample_info: 1495
In sample_info but NOT in expression: 1840
In both: 0


In [17]:
import re

# Check if all headers follow "SYMBOL (ENTREZ_ID)" format
sample_headers = df.columns[:20].tolist()
print("Sample of 20 column headers:")
for h in sample_headers:
    print(" ", h)

# Test the pattern
pattern = re.compile(r'^.+\s\(\d+\)$')
matching = sum(1 for c in df.columns if pattern.match(c))
print(f"\nColumns matching 'SYMBOL (ENTREZ_ID)' format: {matching} / {len(df.columns)}")
print("Columns NOT matching:", len(df.columns) - matching)

# Show any that don't match
non_matching = [c for c in df.columns if not pattern.match(c)]
print("Non-matching examples:", non_matching[:5])

Sample of 20 column headers:
  TSPAN6 (ENSG00000000003)
  TNMD (ENSG00000000005)
  DPM1 (ENSG00000000419)
  SCYL3 (ENSG00000000457)
  C1orf112 (ENSG00000000460)
  FGR (ENSG00000000938)
  CFH (ENSG00000000971)
  FUCA2 (ENSG00000001036)
  GCLC (ENSG00000001084)
  NFYA (ENSG00000001167)
  STPG1 (ENSG00000001460)
  NIPAL3 (ENSG00000001461)
  LAS1L (ENSG00000001497)
  ENPP4 (ENSG00000001561)
  SEMA3F (ENSG00000001617)
  CFTR (ENSG00000001626)
  ANKIB1 (ENSG00000001629)
  CYP51A1 (ENSG00000001630)
  KRIT1 (ENSG00000001631)
  RAD52 (ENSG00000002016)

Columns matching 'SYMBOL (ENTREZ_ID)' format: 0 / 53961
Columns NOT matching: 53961
Non-matching examples: ['TSPAN6 (ENSG00000000003)', 'TNMD (ENSG00000000005)', 'DPM1 (ENSG00000000419)', 'SCYL3 (ENSG00000000457)', 'C1orf112 (ENSG00000000460)']


In [18]:
if egfr_cols:
    egfr = df[egfr_cols[0]]
    print("EGFR expression statistics:")
    print(egfr.describe())
    print("\nTop 10 cell lines expressing EGFR:")
    print(egfr.nlargest(10))
    print("\nBottom 10 cell lines expressing EGFR:")
    print(egfr.nsmallest(10))
    
    # A431 should be high — it's your ground truth
    a431_rows = [idx for idx in df.index if 'A431' in str(idx)]
    print("\nA431 rows found:", a431_rows)

EGFR expression statistics:
count    1495.000000
mean        3.515053
std         2.371441
min         0.000000
25%         0.765535
50%         4.147307
75%         5.322829
max        10.531869
Name: EGFR (ENSG00000146648), dtype: float64

Top 10 cell lines expressing EGFR:
PR-8BsVlg    10.531869
PR-xQ6o5S    10.076148
PR-7ATO6F     9.673645
PR-hK7xqW     9.417620
PR-3H2lE3     9.286535
PR-Vas5QC     9.258024
PR-WRsCFG     9.182568
PR-fPu3e3     8.925347
PR-dMidL0     8.867155
PR-n51AWc     8.362821
Name: EGFR (ENSG00000146648), dtype: float64

Bottom 10 cell lines expressing EGFR:
PR-84JV3M    0.0
PR-CHSpwA    0.0
PR-ap8RKC    0.0
PR-TjzlzI    0.0
PR-swYFJE    0.0
PR-skeXpP    0.0
PR-gErAlu    0.0
PR-Rm18e9    0.0
PR-i3qf08    0.0
PR-JeDpvG    0.0
Name: EGFR (ENSG00000146648), dtype: float64

A431 rows found: []


In [21]:
print(sample_info.columns.tolist())

['DepMap_ID', 'cell_line_name', 'stripped_cell_line_name', 'CCLE_Name', 'alias', 'COSMICID', 'sex', 'source', 'RRID', 'WTSI_Master_Cell_ID', 'sample_collection_site', 'primary_or_metastasis', 'primary_disease', 'Subtype', 'age', 'Sanger_Model_ID', 'depmap_public_comments', 'lineage', 'lineage_subtype', 'lineage_sub_subtype', 'lineage_molecular_subtype', 'default_growth_pattern', 'model_manipulation', 'model_manipulation_details', 'patient_id', 'parent_depmap_id', 'Cellosaurus_NCIt_disease', 'Cellosaurus_NCIt_id', 'Cellosaurus_issues']


In [22]:
a431_info = sample_info[sample_info['cell_line_name'].str.contains('A431', case=False, na=False)]
print("A431 in sample_info:")
print(a431_info[['DepMap_ID', 'cell_line_name', 'primary_disease', 'lineage']].head())

A431 in sample_info:
Empty DataFrame
Columns: [DepMap_ID, cell_line_name, primary_disease, lineage]
Index: []


In [23]:
profiles = pd.read_csv('./data/nomenclature/8_DepMap_OmicsProfiles.csv')
print(profiles.columns.tolist())
print(profiles.head())
print(profiles.shape)

['ProfileID', 'ModelCondition', 'ModelID', 'Datatype', 'WESKit']
   ProfileID  ModelCondition     ModelID Datatype WESKit
0  PR-00UtU3  MC-001131-kkJv  ACH-001131      wgs    NaN
1  PR-01r7OM  MC-000957-Yckn  ACH-000957      rna    NaN
2  PR-02XmLG  MC-002785-qo9e  ACH-002785      rna    NaN
3  PR-04VvBz  MC-001289-BpdI  ACH-001289      wes    ICE
4  PR-09gmEI  MC-000520-YIm7  ACH-000520      rna    NaN
(3830, 5)


In [24]:
no_symbol = [c for c in df.columns if c.startswith('ENSG')]
has_symbol = [c for c in df.columns if not c.startswith('ENSG')]
print("Columns with symbol + Ensembl:", len(has_symbol))
print("Columns with Ensembl only:", len(no_symbol))

# Parse the Ensembl ID extraction pattern
import re
sample = 'TSPAN6 (ENSG00000000003)'
ensembl = re.search(r'(ENSG\d+)', sample).group(1)
print("Extraction test:", ensembl)

Columns with symbol + Ensembl: 53539
Columns with Ensembl only: 422
Extraction test: ENSG00000000003


In [25]:
# 1. Understand the profile duplication question
# Does one cell line have multiple RNA profiles?
rna_profiles = profiles[profiles['Datatype'] == 'rna']
print("Total RNA profiles:", len(rna_profiles))
print("Unique ACH IDs with RNA data:", rna_profiles['ModelID'].nunique())
print("ACH IDs with more than one RNA profile:")
print(rna_profiles['ModelID'].value_counts()[
    rna_profiles['ModelID'].value_counts() > 1].head(10))

Total RNA profiles: 1495
Unique ACH IDs with RNA data: 1479
ACH IDs with more than one RNA profile:
ModelID
ACH-000975    2
ACH-001192    2
ACH-000597    2
ACH-000556    2
ACH-000328    2
ACH-000029    2
ACH-000468    2
ACH-000532    2
ACH-000337    2
ACH-000455    2
Name: count, dtype: int64


In [26]:
# 2. How many of the 1495 profiles are RNA vs other datatypes
print(profiles['Datatype'].value_counts())

Datatype
wes    1860
rna    1495
wgs     475
Name: count, dtype: int64
